# 08 — HITL Capstone: The Full DeepBrief

All the pieces are built. We have:

| From notebook | We built |
|---|---|
| 01-03 | Bounded ReAct agent loop with cost & loop guards |
| 04-05 | Two MCP servers (notes, cache) on stdio + Streamable HTTP |
| 06    | Adapter that lets the agent treat MCP tools and native tools the same |
| 07    | Coordinator + parallel researchers via A2A |

**One thing missing.** Right now the system writes a brief and saves it. That's a Tier-2 action — irreversible, customer-facing, our name is on it. Production agents do NOT auto-publish. They route to a **human-in-the-loop editor** for sync approval.

**By the end of this notebook you will:**
1. Build a durable approval queue (survives process restarts)
2. Wire the editor as a sync gate between synthesis and save
3. Approve via inline `ipywidgets` buttons — that's the production UX shape
4. Run the **complete DeepBrief** end-to-end and inspect the saved brief

Lecture reference: **S8 §5.5** (HITL approval gates).


## 1. The 4-tier HITL taxonomy

| Tier | Risk class | Examples | Mode |
|---|---|---|---|
| **0** | Read-only, idempotent | DB read, web search, weather | **Auto** — log only |
| **1** | Non-critical writes, low blast radius | Send notification, file ticket | Logged async (audit sample) |
| **2** | Irreversible writes, $ involved | **Save customer brief**, send email, modify staging | **Sync HITL** ← we are here |
| **3** | Regulated / high-stakes | Charge > $X, prod data write, customer-facing comms | Block; multi-approver if needed |

**Senior-level insight:** this taxonomy is not encoded per-agent. It's a **fleet-wide policy**. Every agent in the system is subject to the same tier rules. When a new agent adds a new action type, the policy classification governs it by default — the developer doesn't need to think about it. (See [Waxell, *Human-in-the-Loop vs Human-on-the-Loop*](https://www.waxell.ai/blog/human-in-the-loop-vs-human-on-the-loop-ai-agents).)


## 2. Read the editor

`ApprovalStore` lives at `src/deepbrief/agents/editor.py`. Two design choices:

1. **File-backed** — pending approvals are JSON files in `./data/approvals/pending/`. Decided ones move to `./data/approvals/decided/`. Survives process restart.
2. **`wait_for_decision` polls** — async loop that checks every 500ms. In production you'd use a Redis pub/sub or webhook. The polling pattern works everywhere and is trivial to debug.

In [ ]:
import inspect
from deepbrief.agents.editor import ApprovalStore, ApprovalRequest

src = inspect.getsource(ApprovalStore)
print(src[:2000])
print("... [decide() and wait_for_decision()] ...")

## 3. Start the researchers

Same as notebook 07 — two researcher subprocesses on 9001/9002. If yours from before are still running, the `pkill` line cleans up first.

In [ ]:
import os, subprocess, sys, time

subprocess.run(["pkill", "-f", "deepbrief.agents.researcher"], capture_output=True)
time.sleep(1)

researchers = [
    subprocess.Popen(
        [sys.executable, "-m", "deepbrief.agents.researcher", "--port", str(p)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        env=os.environ.copy(),
    )
    for p in [9001, 9002]
]
time.sleep(3)
for proc, port in zip(researchers, [9001, 9002]):
    print(f":{port} alive={proc.poll() is None}")

## 4. Run the coordinator and produce a draft

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from deepbrief.agents.coordinator import Coordinator

TOPIC = "State of agent-protocol adoption: MCP vs A2A in 2026"

coordinator = Coordinator(researcher_urls=[
    "http://localhost:9001",
    "http://localhost:9002",
])
draft = await coordinator.run(TOPIC)
print(draft.markdown)

## 5. Queue the brief for human approval

In [ ]:
store = ApprovalStore(base_dir="./data/approvals")
req = store.create(topic=TOPIC, draft_markdown=draft.markdown)
print(f"📬 Created request {req.request_id}")
print(f"   pending file: ./data/approvals/pending/{req.request_id}.json")
print(f"   state: {req.state}")

## 6. Approve via ipywidgets (the production UX shape)

In a real product the editor surface is a web UI — Slack DM, internal admin panel, dashboard card. In a notebook we use `ipywidgets` to mimic the same shape: three buttons, one decision.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

approve_btn = widgets.Button(description="✅ Approve", button_style="success")
reject_btn  = widgets.Button(description="❌ Reject", button_style="danger")
edit_btn    = widgets.Button(description="✏️ Edit & approve", button_style="warning")
edit_box    = widgets.Textarea(value=draft.markdown, layout=widgets.Layout(width="100%", height="200px"))
status      = widgets.HTML(value=f"<i>request {req.request_id} — pending</i>")

def _approve(_):
    store.decide(req.request_id, "approved", decided_by="notebook")
    status.value = f"<b>✅ approved</b> ({req.request_id})"

def _reject(_):
    store.decide(req.request_id, "rejected", decided_by="notebook")
    status.value = f"<b>❌ rejected</b> ({req.request_id})"

def _edit(_):
    store.decide(
        req.request_id, "edited",
        decided_by="notebook", final_markdown=edit_box.value,
    )
    status.value = f"<b>✏️ edited & approved</b> ({req.request_id})"

approve_btn.on_click(_approve)
reject_btn.on_click(_reject)
edit_btn.on_click(_edit)

display(widgets.VBox([
    widgets.HTML("<h3>Review the draft, then click one button:</h3>"),
    widgets.HBox([approve_btn, reject_btn, edit_btn]),
    widgets.HTML("<b>Edit field</b> (used only if you click 'Edit & approve'):"),
    edit_box,
    status,
]))

**Click one of the buttons above** before running the next cell. The next cell calls `wait_for_decision` which blocks (polling) until you decide. If the kernel were a worker process it could be killed and restarted between create + decide, and the decision would still be captured because everything is on disk.

In [ ]:
decision = await store.wait_for_decision(req.request_id, timeout_s=300)
print(f"state:       {decision.state}")
print(f"decided_by:  {decision.decided_by}")
print(f"decided_at:  {decision.decided_at}")

## 7. Save the brief (or skip if rejected)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import re

if decision.state == "rejected":
    print("❌ Rejected — nothing saved.")
else:
    briefs_dir = Path("./data/briefs")
    briefs_dir.mkdir(parents=True, exist_ok=True)
    slug = re.sub(r"[^a-z0-9]+", "-", TOPIC.lower()).strip("-")[:50]
    ts = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    out_path = briefs_dir / f"{ts}-{slug}.md"
    out_path.write_text(decision.final_markdown)
    print(f"📄 Saved: {out_path}")
    display(Markdown(decision.final_markdown))

## 8. The whole pipeline as a script

Everything we just did is also wired up in `src/deepbrief/run.py`. From a terminal:

```bash
# Start researchers in two other terminals first:
python -m deepbrief.agents.researcher --port 9001
python -m deepbrief.agents.researcher --port 9002

# Then run the pipeline. CLI prompts for approval.
python -m deepbrief.run "State of WebGPU adoption in 2026"

# Or in CI, skip the gate:
python -m deepbrief.run --auto-approve "a topic"

# Or queue for async review:
python -m deepbrief.run --queue-only "a topic"
```


## 9. What you have now (and what's missing)

**Have:**
- ReAct loop with parallel tool calls + graceful termination (S8 §3, §4)
- Strict mode + structured tool layer (S8 §6)
- Cost / loop / step caps with budget enforcement (S8 §4)
- Two MCP servers, two transports, generic adapter into the tool registry (S7 §3-§5)
- Coordinator + parallel researchers via A2A protocol (S7 §7)
- Tier-2 sync HITL gate with durable approval state (S8 §5.5)

**Missing — extension exercises for you:**
1. **Streaming A2A** (`tasks/sendSubscribe`) — researchers emit progress events the coordinator can show in a UI. Currently we just block.
2. **Real observability** — emit OpenTelemetry GenAI spans (S8 §7.2) to Langfuse instead of in-memory `trace`. ~30 minutes with the `langfuse` Python SDK.
3. **MCP gateway** — put Bifrost or supergateway in front of the MCP servers; centralize auth, rate limit, audit. (S7 §8.)
4. **Loop semantic guard** — embed tool args, cosine-similarity threshold against history. Catches `"weather Paris"` ↔ `"current Paris weather"` drift that the literal hash misses.
5. **Code Mode** — instead of sending all 8+ tool schemas every call, let the coordinator emit Python that *uses* the tools as imports, sandbox-execute it. Bifrost's reported impact: ~50% fewer prompt tokens. (S7 §8.3.)


## 10. Cleanup

In [ ]:
for proc in researchers:
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
print("researchers stopped")

## 11. Final self-check

1. Why is the editor a Tier 2 action and not Tier 1 or Tier 3? Defend the classification.
2. The approval store uses files instead of an in-memory dict. What problem would the in-memory version have under a real worker recycling event?
3. We poll for decisions every 500ms. What's the production-grade alternative? When would polling actually win?
4. Where in the pipeline is the *only* place you'd want to bypass HITL during an outage?
5. You're on call and the editor surface is down. The coordinator has 200 pending briefs. What's your fallback?

## You're done.

You've now built every piece of an industrial agent system in ~700 lines of Python. The patterns scale: VoyageAI's production worker uses the same shape (ReAct + MCP + Kafka + MongoDB + HITL) — just bigger.

**Forward links:**
- S9 — Tool-RAG: when you have 30+ MCP tools, semantic retrieval to pick which 5 schemas to send each turn
- S10 — Stateful agents: cross-session memory, agent checkpointing
- S15 — Multi-model routing: send Tier 0 calls to gpt-4o-mini, Tier 3 to o1

Good luck on interviews. 🚀